In [2]:
import yaml
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Leer credenciales del arhivo YAML
with open("credenciales.yaml", "r") as file:
     config = yaml.safe_load(file)

db = config["mysql"]

# Crear conexión
# Su función es crear un Engine, que es el objeto que SQLAlchemy utiliza para comunicarse con la base de datos.
engine = create_engine(                                     
    f"mysql+pymysql://{db['user']}:{db['password']}@"
    f"{db['host']}:{db['port']}/{db['database']}"
)

##### ANALISIS DE EXPERIMENTOS

##### 7-1 Introduccion

AB Testing:

El AB Testing (o prueba A/B) es una técnica estadística muy utilizada en marketing, desarrollo web, ciencia de datos y 
otras áreas para comparar dos versiones de algo (como una página web, un anuncio o una funcionalidad) y determinar cuál funciona mejor. 

Conceptos básicos del AB Testing

Objetivo:

Comparar dos grupos (A y B) para ver si hay diferencias significativas en una métrica clave (ej.: tasa de conversión, clics, ventas).

Ejemplo:

Versión A (actual): Botón verde en un sitio web.

Versión B (nueva): Botón rojo.

Métrica a comparar: % de usuarios que hacen clic.

Hipótesis:

Hipótesis nula (H₀): No hay diferencia entre A y B.

Hipótesis alternativa (H₁): Hay una diferencia significativa.

Métricas comunes:

Tasa de conversión, retención, tiempo en página, ingresos por usuario, etc.

##### 7-2 Por que si y por que no usar SQL

SQL se usa para extraer los datos y manipularlos, para análisis estadístico se requiere software especializado como R o Python.

##### 7-3 Correlacion vs Causalidad

El principio "correlación no implica causalidad" es uno de los conceptos más importantes en estadística y ciencia de datos. 
Significa que aunque dos variables parezcan estar relacionadas (es decir, cambian juntas), no necesariamente una causa la otra. 

3 Razones por las que correlación ≠ causalidad

1.-Variables confusoras (tercer factor):
	
 Una tercera variable influye en ambas.
	
 Ejemplo:
	
 Correlación entre ventas de sombrillas y tormentas.
	
 La variable confusora es la lluvia (aumenta ambas).
 
 2.-Causalidad inversa:
	
 La dirección de la relación es al revés de lo que se asume.
	
 Ejemplo:
	
 Correlación entre "más policías" y "más crímenes".
	
 En realidad, las ciudades con más crimen contratan más policías.

3.-Pura coincidencia (correlación espuria):
	
 La relación es aleatoria y no tiene lógica.

Ejemplo:

Correlación entre el número de piratas y el calentamiento global.

Para probar que A causa B, se necesitan:

-Experimentos controlados:

Como los A/B tests, donde se aísla la variable de interés (ej.: cambiar solo el color de un botón).

-Métodos estadísticos avanzados:

Modelos de regresión con variables instrumentales.

Análisis de series temporales (para descartar tendencias).

-Teoría sólida:

Un mecanismo lógico que explique por qué A debería causar B.

##### 7-4 Experimentos Binarios

Experimento binario: Buscamos ver si se realizó o no una acción de interés. Ejemplo: Los clientes dan click o no dan click en una página web.

Experimento contínuo: Estudiar una variable cuantitativa. Por ejemplo, cuanto dinero gastó un cliente en mi sitio web.

Como herramientas estadísticas se usará la prueba ji-cuadrada y la prueba t de student

In [3]:
# Trabajaremos con datos de una página web

query = r"""
select
     * 
from
     experimentos.paginaweb;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,user_id,variant,views,clicks,purchased_items,experiment_date,lastpurch_date,country
0,1,control,12,0,4,2020-01-01,2020-01-10,Argentina
1,2,control,3,1,3,2020-01-01,2020-01-31,Chile
2,3,control,12,0,2,2020-01-01,2020-01-18,México
3,4,control,2,0,3,2020-01-01,2020-01-18,Perú
4,5,control,4,0,4,2020-01-01,2020-01-26,México
5,6,control,1,1,1,2020-01-01,2020-01-16,Guatemala
6,7,control,27,1,3,2020-01-01,2020-01-26,Argentina
7,8,control,8,0,1,2020-01-01,2020-01-01,Perú
8,9,control,2,0,0,2020-01-01,2020-01-30,México
9,10,control,4,0,4,2020-01-01,2020-01-31,Argentina


In [4]:
# Agrupar por variable variant y obtener cuantos registros tienen click y cuantos no */

query = r"""
select
    variant,
    count(case when clicks=1 then 1 end) as si_click,
    count(case when clicks=0 then 1 end) as no_click
from
    experimentos.paginaweb
group by
    1;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,variant,si_click,no_click
0,control,32628,32372
1,test,32593,32407


In [5]:
# Agrupando por la variable variant obtener el total de usuario, el total de clicks y el porcentaje de clicks */

query = r"""
select
    variant,
    count(*) as intentos_totales,
    count(case when clicks=1 then 1 end) as num_exitos,
    count(case when clicks=1 then 1 end)/count(*) as porcentaje_exitos
from
    experimentos.paginaweb
group by
    1;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,variant,intentos_totales,num_exitos,porcentaje_exitos
0,control,65000,32628,0.5020
1,test,65000,32593,0.5014


La tasa de clics se calcula como:
$$
\hat p=\frac{\text{número de clics}}{\text{intentos totales}}
$$

Para el grupo de control:
$$
\hat p_C=\frac{32628}{65000}=0.501969
$$

Y para el grupo de prueba:
$$
\hat p_T=\frac{32593}{65000}=0.501431
$$

La diferencia observada es:
$$
\begin{aligned}
\hat p_T-\hat p_C
&=0.501431-0.501969\\
&=-0.000538
\end{aligned}
$$

Esto equivale a una diferencia de aproximadamente −0.0538 puntos porcentuales. La página nueva presenta una tasa de clics ligeramente menor, pero todavía debemos comprobar si esa diferencia puede explicarse por variabilidad aleatoria.

Como queremos saber si el número de clics es distinto, debemos usar una prueba bilateral.

Definimos:
- \(p_C\): probabilidad de hacer clic con la página original.
- \(p_T\): probabilidad de hacer clic con la página nueva.
Las hipótesis son:
$$
H_0:p_C=p_T
$$

$$
H_1:p_C\ne p_T
$$

La hipótesis nula \(H_0\) afirma que ambas páginas tienen la misma probabilidad de generar clics.

La hipótesis alternativa \(H_1\) afirma que las probabilidades son diferentes, ya sea porque la nueva página mejora o empeora la tasa de clics.

Antes de calcular, elegimos un nivel de significancia. Usaremos el habitual:
$$
\alpha=0.05
$$

En este caso usamos la prueba Z para dos proporciones independientes.

¿Por qué?
- La variable de respuesta es binaria: hubo clic o no hubo clic.
- Tenemos dos grupos: control y test.
- Queremos comparar dos proporciones.
- Los tamaños de muestra son grandes: 65,000 observaciones por grupo.
La prueba compara la diferencia observada con la diferencia que esperaríamos encontrar debido al azar si ambas páginas tuvieran la misma tasa de clics.
Calcular el estadístico Z

Bajo la hipótesis nula, calculamos primero la proporción combinada:
$$
\hat p=\frac{x_C+x_T}{n_C+n_T}
$$

Sustituyendo los valores:
$$
\hat p=\frac{32628+32593}{65000+65000}
=\frac{65221}{130000}
\approx 0.5017
$$

El error estándar bajo la hipótesis nula es:
$$
SE_0=\sqrt{\hat p(1-\hat p)
\left(\frac{1}{n_C}+\frac{1}{n_T}\right)}
$$

Y el estadístico de prueba es:
$$
Z=\frac{\hat p_T-\hat p_C}{SE_0}
$$

Al sustituir los datos, obtenemos:
$$
\boxed{Z\approx -0.1941}
$$

El signo negativo indica que la tasa de clics del grupo test fue menor que la del control. Sin embargo, el valor está muy cerca de cero, lo que indica que la diferencia observada es pequeña en relación con la variabilidad esperada.
Ahora calculamos el valor p:

Como la prueba es bilateral, calculamos:
$$
p\text{-valor}=2P(Z\geq |z_{\text{obs}}|)
$$

Para $$(Z=-0.1941)$$, el resultado es:
$$
\boxed{p\text{-valor}\approx 0.8461}
$$

Comparemos este valor con nuestro nivel de significancia:
$$
0.8461>0.05
$$

Por lo tanto, no rechazamos la hipótesis nula.

Esto significa que no encontramos evidencia estadística suficiente para concluir que la tasa de clics sea diferente entre ambas páginas.

##### 7-5.- Experimentos Continuos

Queremos comprobar si los valores promedio del experimento en el grupo de control y en el grupo con el "tratamiento" son significativamente diferentes 

La prueba que usaremos es la T de Student

In [7]:
# Obtener por cada grupo el numero de intentos totales, el promedio de visitas y la desviación estandar */


query = r"""
select
    variant,
    count(*) as tamaño_muestra,
    avg(views) as promedio,
    round(std(views),2) as desviacion_std
from
    experimentos.paginaweb
group by
    1;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,variant,tamaño_muestra,promedio,desviacion_std
0,control,65000,4.9802,5.83
1,test,65000,5.0366,6.35


Usaremos la prueba t de Welch porque permite comparar dos medias sin asumir que las varianzas de ambos grupos son iguales.

Definimos:
$$ \mu_C \text{: promedio poblacional de vistas del grupo control} $$
$$ \mu_T \text{: promedio poblacional de vistas del grupo test} $$
Las hipótesis serían:

$$
H_0: \mu_C = \mu_T
$$

$$
H_1: \mu_C \neq \mu_T
$$

Estamos ante una prueba bilateral, porque queremos saber si los promedios son diferentes, independientemente de cuál sea mayor.

3. Estadístico de prueba
La prueba t de Welch utiliza el siguiente estadístico:
$$
t = \frac{\bar X_C-\bar X_T}
{\sqrt{\frac{s_C^2}{n_C}+\frac{s_T^2}{n_T}}}
$$

Donde:
Símbolo	Significado
$$ \bar X_C \text{= Promedio del grupo control} $$
$$ \bar X_T	\text{= Promedio del grupo test} $$
$$ s_C \text{= 	Desviación estándar del control} $$
$$ s_T	\text{= Desviación estándar del test} $$
$$ n_C	\text{= Tamaño de muestra del control} $$
$$ n_T	\text{= Tamaño de muestra del test} $$

Una ventaja importante es que no se necesita tener los datos individuales, se puede calcular la prueba utilizando únicamente los tamaños de muestra, promedios y desviaciones estándar de cada grupo.


In [11]:
from scipy.stats import ttest_ind_from_stats

# Grupo control
n_control = 65000
mean_control = 4.9802
std_control = 5.83

# Grupo test
n_test = 65000
mean_test = 5.0366
std_test = 6.35

# Prueba t de Welch
resultado = ttest_ind_from_stats(
    mean1=mean_control,
    std1=std_control,
    nobs1=n_control,
    mean2=mean_test,
    std2=std_test,
    nobs2=n_test,
    equal_var=False,
    alternative='two-sided'
)

print("Estadístico t:", resultado.statistic)
print("P-value:", resultado.pvalue)

Estadístico t: -1.668046144241883
P-value: 0.09530899283329977


No rechazamos $H_0 $: No hay evidencia de diferencias significativas entre las medias de ambos grupos.

##### 7-6 Manejando valores sobresalientes.


Windsorizing: El proceso de reemplazar un número específico de valores extremos por valores más pequeños. 

Binarizar: Tranformar la métrica de contínua a binaria.

In [14]:
# Agrupar por tipo de grupo, calcular cuantos usuarios hubo, cuantos clientes compraron, cuantos clientes no compraron */

query = r"""
select
    variant,
    count(distinct(user_id)) as usuarios_totales,
    count(case when purchased_items > 0 then 1 end) as si_compro,
    count(case when purchased_items = 0 then 1 end) as no_compro,
    count(case when purchased_items > 0 then 1 end) / count(distinct(user_id)) as porcentaje_compra
from
    experimentos.paginaweb
group by
    1;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,variant,usuarios_totales,si_compro,no_compro,porcentaje_compra
0,control,65000,56934,8066,0.8759
1,test,65000,56817,8183,0.8741


Usaremos la prueba Z para dos proporciones independientes.

La tasa de compra se calcula así:

$$
\hat p = \frac{\text{Compradores}}{\text{Usuarios totales}}
$$

Por ejemplo, para el grupo control:
$$
\hat p_C = \frac{8066}{65000}=0.8759
$$

Observamos que el grupo test tiene una tasa de compra ligeramente inferior.

La diferencia es:
$$
\hat p_T-\hat p_C = -0.0018
$$

La prueba de hipótesis queda como: 
$$
H_0: p_C=p_T
$$

$$
H_1: p_C\neq p_T
$$

Utilizaremos una prueba bilateral con $ \alpha=0.05 $

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

# Número de compradores
compradores = [56934, 56817]

# Usuarios totales
muestras = [65000, 65000]

# Prueba Z para dos proporciones
z_stat, p_value = proportions_ztest(
    count=compradores,
    nobs=muestras,
    alternative='two-sided'
)

print("Estadístico Z:", z_stat)
print("P-value:", p_value)

Estadístico Z: 0.9812204861209073
P-value: 0.32648402383689823


El p-value es:
$
0.3265 > 0.05
$

Por lo tanto, no rechazamos la hipótesis nula.

No encontramos evidencia estadística suficiente para afirmar que existe una diferencia entre los porcentajes de compra de ambos grupos.

##### 7-7.- Cajas de Tiempo

In [20]:
# Obtener la diferencia entre la fecha de observacion y fecha de ultima compra, el grupo, el usuario, el numero de articulos comprados, solo para 
# aquellos registros donde la fecha de observacion y la fecha de ultima compra difieran a lo mas 15 días 


query = r"""
select
    timestampdiff(day,experiment_date,lastpurch_date) as diferencia_dias,
    variant,
    user_id,
    purchased_items
from
    experimentos.paginaweb
where
    timestampdiff(day,experiment_date,lastpurch_date) <= 15;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,diferencia_dias,variant,user_id,purchased_items
0,9,control,1,4
1,15,control,6,1
2,0,control,8,1
3,7,control,12,1
4,3,control,13,2
5,13,control,14,7
6,9,control,15,0
7,6,control,16,7
8,4,control,18,2
9,7,control,19,3


In [ ]:
# Usando la consulta anterior obtener por tipo de grupo (control y tratamiento) el total de observaciones, el promedio de articulos comprados y 
# la desviacion estandar del numero de articulos comprados.
# Usar la prueba t de student para probar la hipotesis de que la media de los articulos comprados difiere entre ambos grupos. 

quer = r"""
select
    variant, 
    count(user_id) as num_observaciones,
    avg(purchased_items) as prom_articulos,
    round(std(purchased_items),2) as desv_est_articulos
from
    (
    select
        timestampdiff(day,experiment_date,lastpurch_date) as diferencia_dias,
        variant,
        user_id,
        purchased_items
    from
        experimentos.paginaweb
    where
        timestampdiff(day,experiment_date,lastpurch_date) <= 15
    ) as a
group by 1;
"""
df = pd.read_sql(text(quer),engine)
df.head(10)

,variant,num_observaciones,prom_articulos,desv_est_articulos
0,control,33566,3.4945,2.29
1,test,33514,3.4970,2.30


En este caso usamos la prueba t de Welch porque permite comparar dos medias sin asumir que las varianzas de ambos grupos son iguales.

Definimos:
$$ \mu_C \text{: promedio poblacional de compras del grupo control} $$
$$ \mu_T \text{: promedio poblacional de compras del grupo test} $$
Las hipótesis serían:

$$
H_0: \mu_C = \mu_T
$$

$$
H_1: \mu_C \neq \mu_T
$$

Estamos ante una prueba bilateral, porque queremos saber si los promedios son diferentes, independientemente de cuál sea mayor.

3. Estadístico de prueba
La prueba t de Welch utiliza el siguiente estadístico:
$$
t = \frac{\bar X_C-\bar X_T}
{\sqrt{\frac{s_C^2}{n_C}+\frac{s_T^2}{n_T}}}
$$

Donde:
Símbolo	Significado
$$ \bar X_C \text{= Promedio del grupo control} $$
$$ \bar X_T	\text{= Promedio del grupo test} $$
$$ s_C \text{= 	Desviación estándar del control} $$
$$ s_T	\text{= Desviación estándar del test} $$
$$ n_C	\text{= Tamaño de muestra del control} $$
$$ n_T	\text{= Tamaño de muestra del test} $$

Una ventaja importante es que no se necesita tener los datos individuales, se puede calcular la prueba utilizando únicamente los tamaños de muestra, promedios y desviaciones estándar de cada grupo.


In [23]:
from scipy.stats import ttest_ind_from_stats

# Grupo control
n_control = 33566
mean_control = 3.4945
std_control = 2.29

# Grupo test
n_test = 33514	
mean_test = 3.4970
std_test = 2.30

# Prueba t de Welch
resultado = ttest_ind_from_stats(
    mean1=mean_control,
    std1=std_control,
    nobs1=n_control,
    mean2=mean_test,
    std2=std_test,
    nobs2=n_test,
    equal_var=False,
    alternative='two-sided'
)

print("Estadístico t:", resultado.statistic)
print("P-value:", resultado.pvalue)

Estadístico t: -0.14106587123860134
P-value: 0.8878183284175799


Comparamos el p-value con nuestro nivel de significancia:
$
0.8878 > 0.05
$

Por lo tanto, no rechazamos la hipótesis nula.

No encontramos evidencia estadística suficiente para afirmar que existe una diferencia entre el promedio de artículos comprados por individuo en ambos grupos.

##### Análisis Pre-Post: Compara a la misma población antes y después de un cambio

In [26]:
# Realizar un analisis del comportamiento de las compras en la pagina web 5 días antes y 5 días después de día de reyes. 
# Usar una prueba estadistica para probar la hipotesis de que la proporcion de compras es la misma en ambos grupos. */

query = r"""
select
    case
        when experiment_date >= "2020-01-01" and experiment_date <= "2020-01-05" then "pre"
        when experiment_date >= "2020-01-06" and experiment_date <= "2020-01-10" then "post"
    end as grupo,
    count(user_id) as num_usuario,
    count(case when purchased_items > 0 then 1 end) as compras,
    count(case when purchased_items > 0 then 1 end) / count(user_id) as pct_compras
from
    experimentos.paginaweb
where
    experiment_date >= "2020-01-01" and experiment_date <= "2020-01-10"
group by
    1;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,grupo,num_usuario,compras,pct_compras
0,pre,13560,11894,0.8771
1,post,32428,28363,0.8746


Ya que se supone que los usuarios de ambos periodos son grupos independientes, usaremos la prueba Z para dos proporciones (Two-proportion Z-test).

- La variable de interés es binaria: compró o no compró.
- Queremos comparar dos proporciones.
- Tenemos dos grupos de observaciones independientes.

Planteamos las hipótesis:
$$
H_0: p_{pre}=p_{post}
$$

$$
H_1: p_{pre}\neq p_{post}
$$

Utilizaremos un nivel de significancia de:
$$
\alpha=0.05
$$

In [27]:
from statsmodels.stats.proportion import proportions_ztest

# Número de usuarios que compraron
compras = [11894, 28363]

# Número total de usuarios
usuarios = [13560, 32428]

z_stat, p_value = proportions_ztest(
    count=compras,
    nobs=usuarios,
    alternative='two-sided'
)

print("Estadístico Z:", z_stat)
print("P-value:", p_value)

alpha = 0.05

if p_value < alpha:
    print("Existe una diferencia significativa")
else:
    print("No existe evidencia de una diferencia significativa")

Estadístico Z: 0.7381530856856079
P-value: 0.46042142788061113
No existe evidencia de una diferencia significativa


Como $ p\text{-value}=0.4604 > 0.05$, no encontramos evidencia estadística suficiente para afirmar que existe una diferencia entre ambos periodos.

##### 7-9-Analisis Experimento Natural

Experimento natural: Generalmente por razones no intencionales 2 poblaciones son separadas y se observa el comportamiento de ciertas variables en ambas poblaciones.


In [30]:
# Comparar las ventas en Chile y Argentina y usar una prueba estadistica para probar la hipotesis de que el porcentaje de compras es distinto en ambos grupos.


query = r"""
select
    country,
    count(user_id) as usuarios,
    count(case when purchased_items > 0 then 1 end) as compras,
    count(case when purchased_items > 0 then 1 end) / count(user_id) as pct_compras
from
    experimentos.paginaweb
where
    country in ("Chile","Argentina")
group by
    1;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,country,usuarios,compras,pct_compras
0,Argentina,22873,22866,0.9997
1,Chile,26804,22802,0.8507


Ya que suponemos que las observaciones son independientes, usaremos la prueba Z para dos proporciones.

Definimos:

$p_A$: proporción de compras en Argentina.

$p_C$: proporción de compras en Chile.

Las hipótesis son:
$$
H_0: p_A=p_C
$$

$$
H_1: p_A\neq p_C
$$

Utilizamos un nivel de significancia de 5%.

Adicionalmente podemos usar también la prueba exacta de Fisher en caso de que los conteos esperados de alguna de las celdas sea menor que 5.


In [31]:
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import fisher_exact

# Argentina y Chile
compras = [22866, 22802]
usuarios = [22873, 26804]

# Prueba Z
z_stat, p_z = proportions_ztest(
    count=compras,
    nobs=usuarios,
    alternative='two-sided'
)

# Prueba exacta de Fisher
tabla = [
    [22866, 7],
    [22802, 4002]
]

odds_ratio, p_fisher = fisher_exact(
    tabla,
    alternative='two-sided'
)

print("Z:", z_stat)
print("P-value Z:", p_z)
print("P-value Fisher:", p_fisher)

Z: 60.77170194547449
P-value Z: 0.0
P-value Fisher: 0.0


Ambas pruebas nos dan un p-value muy 0, por lo que concluimos que existe evidencia estadística de una diferencia entre las proporciones de compra.